In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cpu


In [ ]:
class RNN(torch.nn.Module):
  def __init__(self, num_inputs, num_hidden, sigma=0.1):
    super().__init__()
    self.num_inputs = num_inputs
    self.num_hidden = num_hidden
    self.W_xh = torch.nn.Parameter(torch.randn(num_inputs, num_hidden) * sigma)
    self.W_hh = torch.nn.Parameter(torch.randn(num_hidden, num_hidden) * sigma)
    self.b_h = torch.nn.Parameter(torch.zeros(num_hidden))
  def forward(self, inputs, state=None):
    if state is None:
      state = torch.zeros((inputs.shape[1], self.num_hidden), device=self.W_xh.device)
    else:
      state, = state
    outputs = []
    for x in inputs:
      state = torch.tanh(x @ self.W_xh + state @ self.W_hh + self.b_h)
      outputs.append(state)
    return outputs, state

In [ ]:
batch_size, num_inputs, num_hiddens, num_steps = 2, 16, 32, 100
rnn = RNN(num_inputs, num_hiddens).to(device)
X = torch.ones((num_steps, batch_size, num_inputs)).to(device)
outputs, state = rnn(X)

In [ ]:
def check_len(a, n):
    """Check the length of a list."""
    assert len(a) == n, f'list\'s length {len(a)} != expected length {n}'

def check_shape(a, shape):
    """Check the shape of a tensor."""
    assert a.shape == shape, \
            f'tensor\'s shape {a.shape} != expected shape {shape}'

check_len(outputs, num_steps)
check_shape(outputs[0], (batch_size, num_hiddens))
check_shape(state, (batch_size, num_hiddens))

In [ ]:
class RNNLM(torch.nn.Module):
  def __init__(self, vocab_size:int, rnn: RNN, lr=0.01):
    super().__init__()
    self.vocab_size = vocab_size
    self.rnn = rnn
    self.linear = torch.nn.LazyLinear(self.vocab_size)
    self.loss = torch.nn.CrossEntropyLoss()

  def training_step(self, batch):
    preds = self(*batch[:-1])
    targets = batch[-1]
    l = self.loss(preds.permute(0, 2, 1), batch[-1])
    return l

  def validation_step(self, batch):
    preds = self(*batch[:-1])
    targets = batch[-1]
    l = self.loss(preds.permute(0, 2, 1), batch[-1])
    return l

  def one_hot(self, x):
    return torch.nn.functional.one_hot(x.T, self.vocab_size).type(torch.float32)

  def output_layer(self, hiddens):
    return self.linear(rnn_outputs).swapaxes(0, 1)

  def forward(self, inputs, state=None):
    rnn_outputs, state = self.rnn(self.one_hot(inputs.to(self.W_q.device)), state)
    return self.output_layer(rnn_outputs)

  def predict(self, prefix, num_preds, device=None):
    state, outputs = None, [ord(prefix[0])]
    for i in range(len(prefix) + num_preds - 1):
        X = torch.tensor([[outputs[-1]]], device=device)
        embs = self.one_hot(X)
        rnn_outputs, state = self.rnn(embs, state)
        if i < len(prefix) - 1:  # Warm-up period
            outputs.append(ord(prefix[i + 1]))
        else:  # Predict num_preds steps
            Y = self.output_layer(rnn_outputs)
            outputs.append(int(Y.argmax(2).reshape(1)))
    return ''.join([chr(i) for i in outputs])

In [ ]:
model = RNNLM(num_inputs, rnn).to(device)
outputs = model(torch.ones((batch_size, num_steps), dtype=torch.int64).to(device))
outputs

tensor([[[ 0.8795,  2.5004,  1.3496,  ...,  0.0129,  1.9552,  0.2296],
         [ 1.2682,  1.0358,  1.0306,  ..., -0.1755,  1.8481, -0.5033],
         [ 1.1275,  1.1134,  1.1285,  ..., -0.4386,  1.6205, -0.5645],
         ...,
         [ 0.8482,  1.0342,  0.7546,  ..., -0.2785,  1.1293, -0.2288],
         [ 0.8482,  1.0342,  0.7546,  ..., -0.2785,  1.1293, -0.2288],
         [ 0.8482,  1.0342,  0.7546,  ..., -0.2785,  1.1293, -0.2288]],

        [[ 0.8795,  2.5004,  1.3496,  ...,  0.0129,  1.9552,  0.2296],
         [ 1.2682,  1.0358,  1.0306,  ..., -0.1755,  1.8481, -0.5033],
         [ 1.1275,  1.1134,  1.1285,  ..., -0.4386,  1.6205, -0.5645],
         ...,
         [ 0.8482,  1.0342,  0.7546,  ..., -0.2785,  1.1293, -0.2288],
         [ 0.8482,  1.0342,  0.7546,  ..., -0.2785,  1.1293, -0.2288],
         [ 0.8482,  1.0342,  0.7546,  ..., -0.2785,  1.1293, -0.2288]]],
       grad_fn=<StackBackward0>)

In [ ]:
time_machine_url = 'https://www.gutenberg.org/cache/epub/35/pg35.txt'

In [ ]:
import requests

In [ ]:
time_machine_text = requests.get(time_machine_url).text

In [ ]:
time_machine_text = time_machine_text.encode('ascii', 'ignore')

In [ ]:
time_machine_list = [int(c) for c in time_machine_text]

In [ ]:
num_steps = 32

In [ ]:
input_tensor, output_tensor = torch.stack([torch.tensor(time_machine_list[i:len(time_machine_list) - num_steps + i]) for i in range(num_steps)], 1), torch.stack([torch.tensor(time_machine_list[i + 1:len(time_machine_list) - num_steps + i + 1]) for i in range(num_steps)], 1)

In [ ]:
input_tensor[0]

tensor([ 84, 104, 101,  32,  80, 114, 111, 106, 101,  99, 116,  32,  71, 117,
        116, 101, 110,  98, 101, 114, 103,  32, 101,  66, 111, 111, 107,  32,
        111, 102,  32,  84])

In [ ]:
output_tensor[0]

tensor([104, 101,  32,  80, 114, 111, 106, 101,  99, 116,  32,  71, 117, 116,
        101, 110,  98, 101, 114, 103,  32, 101,  66, 111, 111, 107,  32, 111,
        102,  32,  84, 104])

In [ ]:
dataset = torch.utils.data.dataset.TensorDataset(input_tensor, output_tensor)

In [ ]:
dataloader = torch.utils.data.dataloader.DataLoader(dataset, batch_size=1024)

In [ ]:
num_inputs = 123

In [ ]:
def train_rnn(num_inputs, num_hiddens, num_steps, lr, max_epochs, max_grad):
  input_tensor, output_tensor = torch.stack([torch.tensor(time_machine_list[i:len(time_machine_list) - num_steps + i]) for i in range(num_steps)], 1), torch.stack([torch.tensor(time_machine_list[i + 1:len(time_machine_list) - num_steps + i + 1]) for i in range(num_steps)], 1)
  dataset = torch.utils.data.dataset.TensorDataset(input_tensor, output_tensor)
  dataloader = torch.utils.data.dataloader.DataLoader(dataset, batch_size=1024, shuffle=True)
  rnn = RNN(num_inputs, num_hiddens)
  model = RNNLM(num_inputs, rnn).to(device)
  optim = torch.optim.SGD(model.parameters(), lr=lr)
  for epoch in range(1, max_epochs + 1):
    total_loss = 0
    num_batches = 0
    for X, y in dataloader:
      # Move batch data to the device
      X, y = X.to(device), y.to(device)

      optim.zero_grad()
      # `X` will now be `(inputs_batch, targets_batch)` from the dataloader.
      # The `training_step` expects `batch` to be `(inputs, targets)` for `self(*batch[:-1]), batch[-1])`
      # So, `model.training_step((X, y))` or `model.training_step(X, y)` is needed if it takes separate args.
      # Given `l = self.loss(self(*batch[:-1]), batch[-1])`, `batch` should be `(X, y)`.
      # So, `model.training_step((X, y))` will correctly unpack as `*batch[:-1]` -> `X` and `batch[-1]` -> `y`.
      l = model.training_step((X, y))
      l.backward()
      # parameter_list = [p for p in model.parameters() if p.requires_grad]
      # norm = torch.sqrt(sum(torch.sum(p.grad ** 2) for p in parameter_list))
      # if norm > max_grad:
      #   for p in parameter_list:
      #     p.grad[:] = p.grad[:] / norm * max_grad # Correct gradient clipping: scale by max_grad/norm
      optim.step()
      total_loss += l.item()
      num_batches += 1
      if num_batches >= 15:
        break
    avg_loss = total_loss / num_batches if num_batches > 0 else 0
    print(f"Epoch {epoch}, Total Train Loss: {total_loss:.4f}, Average Train Loss: {avg_loss:.4f}")
  return model

In [ ]:
num_hiddens = 32
num_steps =  100
lr = 0.5
grad_max = 1
max_epochs = 10

In [ ]:
model = train_rnn(num_inputs, num_hiddens, num_steps, lr, max_epochs, grad_max)

Epoch 1, Total Train Loss: 64.6242, Average Train Loss: 4.3083
Epoch 2, Total Train Loss: 51.8143, Average Train Loss: 3.4543
Epoch 3, Total Train Loss: 46.7909, Average Train Loss: 3.1194
Epoch 4, Total Train Loss: 43.8988, Average Train Loss: 2.9266
Epoch 5, Total Train Loss: 42.4951, Average Train Loss: 2.8330
Epoch 6, Total Train Loss: 41.2613, Average Train Loss: 2.7508
Epoch 7, Total Train Loss: 40.4459, Average Train Loss: 2.6964
Epoch 8, Total Train Loss: 39.6053, Average Train Loss: 2.6404
Epoch 9, Total Train Loss: 39.0374, Average Train Loss: 2.6025
Epoch 10, Total Train Loss: 38.9031, Average Train Loss: 2.5935


In [ ]:
model.predict('is has', 20, device)

'is has thenthenthenthenthe'

In [ ]:
war_of_worlds_text = requests.get('https://www.gutenberg.org/cache/epub/36/pg36.txt').text

In [ ]:
war_of_worlds_text = list(war_of_worlds_text.encode('ascii', 'ignore'))

In [ ]:
input_tensor, output_tensor = torch.stack([torch.tensor(war_of_worlds_text[i:len(war_of_worlds_text) - num_steps + i]) for i in range(num_steps)], 1), torch.stack([torch.tensor(war_of_worlds_text[i + 1:len(war_of_worlds_text) - num_steps + i + 1]) for i in range(num_steps)], 1)

In [ ]:
war_of_worlds_dataset = torch.utils.data.dataset.TensorDataset(input_tensor, output_tensor)

In [ ]:
val_dataset = torch.utils.data.dataloader.DataLoader(war_of_worlds_dataset, batch_size=1024)

In [ ]:
model.eval()
total_loss = 0
num_batches = 0
for X, y in val_dataset:
  X, y = X.to(device), y.to(device)
  l = model.validation_step((X, y))
  num_batches += 1
  total_loss += l.item()
avg_loss = total_loss / num_batches

In [ ]:
max(war_of_worlds_text)

122

In [ ]:
avg_loss

2.5531966241739563

In [ ]:
tale_text = requests.get('https://www.gutenberg.org/ebooks/98.txt.utf-8').text

In [ ]:
tale_list = list(tale_text.encode('ascii', 'ignore'))

In [ ]:
max(tale_list)

122

In [ ]:
input_tensor, output_tensor = torch.stack([torch.tensor(tale_list[i:len(tale_list) - num_steps + i]) for i in range(num_steps)], 1), torch.stack([torch.tensor(tale_list[i + 1:len(tale_list) - num_steps + i + 1]) for i in range(num_steps)], 1)

In [ ]:
tale_dataset = torch.utils.data.dataset.TensorDataset(input_tensor, output_tensor)

In [ ]:
tale_dataloader = torch.utils.data.dataloader.DataLoader(tale_dataset, batch_size=1024)

In [ ]:
model.eval()
total_loss = 0
num_batches = 0
for X, y in tale_dataloader:
  X, y = X.to(device), y.to(device)
  l = model.validation_step((X, y))
  num_batches += 1
  total_loss += l.item()
avg_loss = total_loss / num_batches

In [ ]:
avg_loss

2.5830744029954076

In [ ]:
def predict(self, prefix, num_preds, device=None, alpha=1.03):
  state, outputs = None, [ord(prefix[0])]
  for i in range(len(prefix) + num_preds - 1):
      X = torch.tensor([[outputs[-1]]], device=device)
      embs = self.one_hot(X)
      rnn_outputs, state = self.rnn(embs, state)
      if i < len(prefix) - 1:  # Warm-up period
          outputs.append(ord(prefix[i + 1]))
      else:  # Predict num_preds steps
          Y = self.output_layer(rnn_outputs) # Y has shape (1, 1, vocab_size)
          # Squeeze Y to get a 1D tensor of logits (vocab_size,)
          logits = Y.squeeze()
          # Apply alpha as an inverse temperature for sampling (alpha > 1 makes distribution sharper)
          sampled_idx = torch.distributions.Categorical(logits=logits * alpha).sample()
          outputs.append(sampled_idx.item()) # .item() to convert 0-dim tensor to scalar
  return ''.join([chr(i) for i in outputs])

In [ ]:
predict(model, 'it has', 20, device, 10)

'it has me the  on the s on'